## Find tables

In [ ]:
%%sql -r dataframe_6
SHOW TABLES IN DATABASE US_OPEN_CENSUS_DATA_NEIGHBORHOOD_INSIGHTS_FREE_DATASET;


### Metadata analysis

In [ ]:
%%sql -r dataframe_5
DESCRIBE TABLE
US_OPEN_CENSUS_DATA_NEIGHBORHOOD_INSIGHTS_FREE_DATASET.PUBLIC."2019_METADATA_CBG_FIELD_DESCRIPTIONS";

In [ ]:
%%sql -r dataframe_7
SELECT *
FROM US_OPEN_CENSUS_DATA_NEIGHBORHOOD_INSIGHTS_FREE_DATASET.PUBLIC."2019_METADATA_CBG_FIELD_DESCRIPTIONS"
LIMIT 10;

## DB schema -> JSON

### Constants + session

In [ ]:
from snowflake.snowpark.context import get_active_session
from collections import defaultdict
import json


session = get_active_session()

DB = "US_OPEN_CENSUS_DATA_NEIGHBORHOOD_INSIGHTS_FREE_DATASET"
SCHEMA = "PUBLIC"
YEAR = 2019


### Load the metadata

In [ ]:
metadata = session.sql(f"""
    SELECT *
    FROM {DB}.{SCHEMA}."2019_METADATA_CBG_FIELD_DESCRIPTIONS"
""").to_pandas()

print(f"Metadata rows: {len(metadata):,}")
print(metadata.columns.tolist())

In [ ]:
columns = session.sql(f"""
    SELECT
        TABLE_NAME,
        COLUMN_NAME,
        DATA_TYPE,
        ORDINAL_POSITION
    FROM {DB}.INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = '{SCHEMA}'
      AND TABLE_NAME LIKE '{YEAR}_CBG_%'
    ORDER BY TABLE_NAME, ORDINAL_POSITION
""").to_pandas()

print(f"Physical columns: {len(columns):,}")
print(f"Physical tables:  {columns['TABLE_NAME'].nunique():,}")

In [ ]:
columns.head(20)

### Mapping - physical idx


In [ ]:
physical_index = defaultdict(list)

for _, row in columns.iterrows():
    column_name = row["COLUMN_NAME"].upper()

    physical_index[column_name].append({
        "physical_table": row["TABLE_NAME"],
        "physical_column": row["COLUMN_NAME"],
        "data_type": row["DATA_TYPE"],
        "ordinal_position": int(row["ORDINAL_POSITION"]),
    })

In [ ]:
physical_index["B08303E12"], physical_index["B19013E1"]

## Semmantic path

In [ ]:
FIELD_COLUMNS = [
    "FIELD_LEVEL_1",
    "FIELD_LEVEL_2",
    "FIELD_LEVEL_3",
    "FIELD_LEVEL_4",
    "FIELD_LEVEL_5",
    "FIELD_LEVEL_6",
    "FIELD_LEVEL_7",
    "FIELD_LEVEL_8",
    "FIELD_LEVELl_9",   # typo в исходной таблице
    "FIELD_LEVEL_10",
]


def build_semantic_path(row):
    path = []

    for column in FIELD_COLUMNS:
        value = row.get(column)

        if value is not None:
            value = str(value).strip()

            if value:
                path.append(value)

    return path



In [ ]:
catalog = []

unmatched = []
ambiguous = []
duplicate_metadata = []

seen_ids = set()

for _, row in metadata.iterrows():

    table_id = str(row["TABLE_ID"]).strip()
    normalized_id = table_id.upper()

    # -----------------------------------------
    # 1. Duplicate metadata check
    # -----------------------------------------

    if normalized_id in seen_ids:
        duplicate_metadata.append(table_id)

    seen_ids.add(normalized_id)

    # -----------------------------------------
    # 2. Find physical column
    # -----------------------------------------

    matches = physical_index.get(normalized_id, [])

    if len(matches) == 0:
        unmatched.append(table_id)
        continue

    if len(matches) > 1:
        ambiguous.append({
            "table_id": table_id,
            "matches": matches,
        })
        continue

    physical = matches[0]

    # -----------------------------------------
    # 3. Semantic path
    # -----------------------------------------

    semantic_path = build_semantic_path(row)

    # -----------------------------------------
    # 4. Build catalog record
    # -----------------------------------------

    record = {
        "year": YEAR,

        "table_id": table_id,
        "table_number": row["TABLE_NUMBER"],

        "physical_table": physical["physical_table"],
        "physical_column": physical["physical_column"],

        "data_type": physical["data_type"],

        "measure_type": (
            str(row["FIELD_LEVEL_1"]).strip().lower()
            if row["FIELD_LEVEL_1"] is not None
            else None
        ),

        "title": row["TABLE_TITLE"],
        "topics": row["TABLE_TOPICS"],
        "universe": row["TABLE_UNIVERSE"],

        "semantic_path": semantic_path,
    }

    catalog.append(record)

## Validation

In [ ]:
print("========== CATALOG VALIDATION ==========")

print(f"Metadata variables:       {len(metadata):,}")
print(f"Catalog records:           {len(catalog):,}")
print(f"Unmatched variables:       {len(unmatched):,}")
print(f"Ambiguous mappings:        {len(ambiguous):,}")
print(f"Duplicate metadata IDs:    {len(duplicate_metadata):,}")

In [ ]:
examples = [
    "B01001e10",
    "B19013e1",
]

for variable_id in examples:

    matches = [
        x for x in catalog
        if x["table_id"].upper() == variable_id.upper()
    ]

    print(json.dumps(matches, indent=2, ensure_ascii=False))

## JSON export


In [ ]:
with open("catalog_2019.json", "w", encoding="utf-8") as f:
    json.dump(
        catalog,
        f,
        indent=2,
        ensure_ascii=False
    )

print(f"Saved {len(catalog):,} variables to catalog_2019.json")